In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CLEAN_PATH = Path("../data/processed/netflix_clean.csv")
DASH_DIR = Path("../reports")
DASH_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_PATH, parse_dates=["date_added"])
df["primary_genre"] = df["listed_in"].str.split(",").str[0].str.strip()

# Headline numbers for the dashboard
kpis = {
    "Total titles": len(df),
    "Movies": int((df["type"] == "Movie").sum()),
    "TV Shows": int((df["type"] == "TV Show").sum()),
    "Countries": int(df.loc[df["primary_country"] != "Unknown", "primary_country"].nunique()),
    "Peak year added": int(df["year_added"].value_counts().idxmax()),
}
kpis

{'Total titles': 8790,
 'Movies': 6126,
 'TV Shows': 2664,
 'Countries': 85,
 'Peak year added': 2019}

In [4]:
RED = "#E50914"
BG, CARD, TEXT, MUTED = "#141414", "#1f1f1f", "#f5f5f1", "#9a9a9a"
TYPE_COLORS = {"Movie": RED, "TV Show": "#00b4d8"}


def style(fig: go.Figure, height: int = 380) -> go.Figure:
    """Apply the shared dark theme to a Plotly figure."""
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(family="Segoe UI, Arial", color=TEXT, size=12),
        title=dict(font=dict(size=16), x=0.02),
        margin=dict(l=40, r=30, t=60, b=40),
        height=height,
        legend=dict(title=None, bgcolor="rgba(0,0,0,0)"),
        bargap=0.25,
    )
    fig.update_xaxes(gridcolor="#2e2e2e", zeroline=False)
    fig.update_yaxes(gridcolor="#2e2e2e", zeroline=False)
    return fig


# 1. Titles added per year, by type
added = df.groupby(["year_added", "type"], observed=True).size().reset_index(name="titles")
fig_growth = px.bar(added, x="year_added", y="titles", color="type", barmode="stack",
                    color_discrete_map=TYPE_COLORS, title="Titles added per year",
                    labels={"year_added": "Year added", "titles": "Titles"})

# 2. Top 10 countries (color scales with value)
countries = (df[df["primary_country"] != "Unknown"]["primary_country"]
             .value_counts().head(10).reset_index())
countries.columns = ["country", "titles"]
fig_countries = px.bar(countries.sort_values("titles"), x="titles", y="country",
                       orientation="h", title="Top 10 countries",
                       color="titles", color_continuous_scale="YlOrRd", text="titles")
fig_countries.update_traces(textposition="outside", cliponaxis=False)
fig_countries.update_coloraxes(showscale=False)

# 3. Top 10 genres (color scales with value)
genres = (df.assign(genre=df["listed_in"].str.split(","))
          .explode("genre").assign(genre=lambda d: d["genre"].str.strip()))
top_genres = genres["genre"].value_counts().head(10).reset_index()
top_genres.columns = ["genre", "titles"]
fig_genres = px.bar(top_genres.sort_values("titles"), x="titles", y="genre",
                    orientation="h", title="Top 10 genres",
                    color="titles", color_continuous_scale="Tealgrn", text="titles")
fig_genres.update_traces(textposition="outside", cliponaxis=False)
fig_genres.update_coloraxes(showscale=False)

# 4. Rating mix: group rare ratings into "Other" so labels do not overlap
ratings = df["rating"].value_counts()
small = ratings[ratings < 100].index
ratings = ratings.rename(lambda r: "Other" if r in small else r).groupby(level=0).sum()
ratings = ratings.sort_values(ascending=False).reset_index()
ratings.columns = ["rating", "titles"]
fig_ratings = px.pie(ratings, names="rating", values="titles", hole=0.55,
                     title="Rating distribution",
                     color_discrete_sequence=px.colors.qualitative.Bold)
fig_ratings.update_traces(textposition="inside", textinfo="percent",
                          marker=dict(line=dict(color=CARD, width=2)))

# 5. Movies vs TV Shows share (donut)
type_counts = df["type"].value_counts().reset_index()
type_counts.columns = ["type", "titles"]
fig_type = px.pie(type_counts, names="type", values="titles", hole=0.55,
                  title="Movies vs TV Shows", color="type",
                  color_discrete_map=TYPE_COLORS)
fig_type.update_traces(textinfo="percent+label",
                       marker=dict(line=dict(color=CARD, width=2)))

charts = [style(f) for f in (fig_growth, fig_type, fig_countries, fig_genres, fig_ratings)]

kpi_html = "".join(
    f'<div class="kpi"><div class="v">{v if k == "Peak year added" else f"{v:,}"}</div>'
    f'<div class="l">{k}</div></div>'
    for k, v in kpis.items()
)

chart_html = "".join(
    f'<div class="card">{f.to_html(full_html=False, include_plotlyjs=True if i == 0 else False)}</div>'
    for i, f in enumerate(charts)
)

html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Netflix Business Insights Dashboard</title>
<style>
  * {{ box-sizing: border-box; }}
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background:{BG}; color:{TEXT};
         margin:0; padding:32px; }}
  header {{ border-left:6px solid {RED}; padding-left:16px; margin-bottom:28px; }}
  h1 {{ margin:0; font-size:30px; letter-spacing:.5px; }}
  .sub {{ color:{MUTED}; margin:6px 0 0; }}
  .kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr));
           gap:16px; margin-bottom:24px; }}
  .kpi {{ background:{CARD}; border-radius:12px; padding:20px 24px;
          border-top:3px solid {RED}; transition:transform .15s; }}
  .kpi:hover {{ transform:translateY(-3px); }}
  .kpi .v {{ font-size:32px; font-weight:700; color:{TEXT}; }}
  .kpi .l {{ color:{MUTED}; margin-top:4px; text-transform:uppercase;
             font-size:12px; letter-spacing:1px; }}
  .grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(480px,1fr)); gap:16px; }}
  .card {{ background:{CARD}; border-radius:12px; padding:8px; overflow:hidden; }}
  footer {{ color:{MUTED}; text-align:center; margin-top:28px; font-size:12px; }}
</style></head><body>
<header>
  <h1>Netflix Business Insights Dashboard</h1>
  <p class="sub">Auspify Data Science Internship - Task 6</p>
</header>
<div class="kpis">{kpi_html}</div>
<div class="grid">{chart_html}</div>
<footer>Source: Netflix titles dataset (8,790 titles), titles added up to September 2021</footer>
</body></html>"""

out = DASH_DIR / "dashboard.html"
out.write_text(html, encoding="utf-8")
print("Saved:", out.resolve())

Saved: C:\Users\HP\auspify-data-science-internship\reports\dashboard.html


## Business Insights and Recommendations

**Key insights**

1. **Movies dominate the catalogue** (6,126 titles, 69.7%) while TV shows are 30.3%. TV shows are the smaller but steadily growing share each year.
2. **The US is the largest source of content** (3,240 titles), followed by India (1,057) and the UK (638). India and Pakistan together show strong regional supply.
3. **International Movies, Dramas and Comedies** are the three biggest genres (2,752 / 2,426 / 1,674 titles).
4. **Mature content is the biggest audience segment:** TV-MA (36.5%) and TV-14 (24.5%) make up about 61% of titles, while Kids content (TV-Y, TV-Y7, TV-G, G, PG, TV-PG) is a much smaller share.
5. **Additions peaked in 2019** (2,016 titles) and have since slowed, with 2021 incomplete (data ends September 2021). Release-year output peaked in 2018 and has declined since.
6. **Audience group can be predicted from metadata** with about 68% accuracy (Gradient Boosting vs 46% baseline). Kids content is identified precisely (precision 0.85), Teens is hardest to separate from Mature.

**Recommendations**

1. **Invest in Kids and Family content.** It is under-represented compared to Mature content, so it is a gap for family subscribers.
2. **Grow TV shows.** Series drive repeat viewing; only 30% of the catalogue is TV, and most are single-season, so renewing successful shows could increase retention.
3. **Keep expanding regional content** (India, Pakistan, South Korea, Japan) where supply is growing and genres like International Movies are already among the top.
4. **Fix metadata quality.** 2,588 titles (29%) have no director and 287 no country; better metadata would improve search, recommendations and any prediction model.
5. **Add plot descriptions and cast data** to improve the recommender (Task 3) and the audience classifier (Task 5).
6. **Do not rely on linear trend forecasts.** Output has fallen since the 2018 peak, so growth planning should use richer data than release year alone.

**Limitations:** the dataset has no viewership, revenue or user ratings, so these recommendations describe the catalogue, not viewer demand.